# Encrypted Inference API — End-to-End Demo

This notebook demonstrates the full encrypted inference workflow using the
Encrypted Inference API reference implementation and Python SDK.

It walks through:

1. Discovering available models
2. Inspecting model metadata
3. Building a compatible CKKS session locally
4. Encrypting client input locally
5. Submitting encrypted inference to the server
6. Polling for the result
7. Decrypting the result locally

Plaintext inputs and the secret key remain client-side.
The server receives encrypted payloads only.

## Preconditions

Before running this notebook:

1. The reference server is running locally at `http://127.0.0.1:8000`
2. The SDK is installed in the current environment
3. Pyfhel and other client dependencies are installed

Example server startup:

```bash
uvicorn server.app.main:app --reload
```

Example install: ``` pip install -e ./client ```

In [ ]:
### Cell 3 — Code
import requests
from pprint import pprint

from heapi_client.client import Client
from heapi_client.ckks.session import CKKS_Session


In [ ]:
BASE_URL = "http://127.0.0.1:8000"
client = Client(base_url=BASE_URL)

## Check server connectivity

In [ ]:
def check_server(base_url: str) -> None:
    try:
        response = requests.get(f"{base_url}/models", timeout=5)
        response.raise_for_status()
        print("Server reachable.")
    except Exception as exc:
        raise RuntimeError(
            f"Could not reach the reference server at {base_url}. "
            "Start the server before running this notebook."
        ) from exc

check_server(BASE_URL)

## Discover available models

The client fetches `/models` and validates the response against the public
model discovery schema before returning it.

In [ ]:
models_payload = client.discovery.list_models()
print("API version:", models_payload["api_version"])
print("Number of models:", len(models_payload["models"]))

In [ ]:
models = models_payload["models"]
if not models:
    raise RuntimeError("No models were returned by the server.")

print("Available models:")
for model in models:
    print("-", model["model_id"], f"(v{model['version']})")

## Select a model and inspect metadata

To keep the demo runnable across environments, this notebook selects the
first discovered model instead of hardcoding a model id.

In [ ]:
model = models[0]
model_id = model["model_id"]
version = model["version"]

print("Selected model:", model_id)
print("Version:", version)
print("Description:", model["description"])
print("HE scheme:", model["he_scheme"])

In [ ]:
enc = model["encryption_parameters"]
inf = model["inference"]
constraints = model["constraints"]
activation = model["activation_parameters"]

print("Encryption parameters")
print("  poly_modulus_degree:", enc["poly_modulus_degree"])
print("  coeff_modulus_bits:", enc["coeff_modulus_bits"])
print("  scale:", enc["scale"])
print("  max_multiplicative_depth:", enc["max_multiplicative_depth"])

print("\nInference metadata")
print("  input_dimension:", inf["input_dimension"])
print("  output_dimension:", inf["output_dimension"])
print("  activation:", inf["activation"])

print("\nActivation parameters")
print("  kind:", activation["kind"])
print("  coefficients:", activation["coefficients"])

print("\nConstraints")
print("  max_batch_size:", constraints["max_batch_size"])

## Build a compatible CKKS session locally

The session is created from model metadata returned by discovery.

In [ ]:
session = CKKS_Session.from_model(model)
print("CKKS session initialized successfully.")

## Prepare a sample input

The client validates the input length against `inference.input_dimension`.
This example uses a single sample, so the batch size is 1.

In [ ]:
input_dimension = model["inference"]["input_dimension"]
sample_input = [float(i + 1) for i in range(input_dimension)]

print("Sample input:")
print(sample_input)

## Encrypt the input locally

The client encrypts one ciphertext per feature column.
For a single sample, this produces one encrypted value per input feature.

In [ ]:
batch = [sample_input]
ciphertexts = session.encrypt_feature_batch(batch)

print("Number of ciphertexts produced:", len(ciphertexts))
print("First ciphertext keys:", list(ciphertexts[0].keys()))
print("First ciphertext encoding:", ciphertexts[0]["encoding"])
print("First ciphertext payload preview:", ciphertexts[0]["payload"][:64] + "...")

## Submit encrypted inference

The request includes:

- `model_id`
- `version`
- encrypted `inputs`
- `batch_size`

In [ ]:
job = client.infer_api.submit(
    model_id=model_id,
    version=version,
    inputs=ciphertexts,
    batch_size=len(batch),
)

print("Submission response:")
pprint(job)

## Poll job status and retrieve the final encrypted response

In [ ]:
job_id = job["job_id"]
result = client.jobs.wait(job_id, interval=1.0, timeout=60.0)

print("Final response:")
pprint(result)

## Decrypt the result locally

The final response contains an encrypted `payload`, which is decrypted
client-side using the CKKS session.## Decrypt the result locally

In [ ]:
decrypted_output = session.decrypt_slots(result, batch_size=len(batch))

print("Decrypted output:")
print(decrypted_output)

In [ ]:
diagnostics = result.get("diagnostics")
if diagnostics:
    print("Diagnostics:")
    pprint(diagnostics)

## Summary

This notebook demonstrated the full happy path:

1. Discover available models
2. Inspect model metadata
3. Build a compatible CKKS session locally
4. Encrypt the client input
5. Submit encrypted inference
6. Poll for the final result
7. Decrypt the result locally

This reinforces the main protocol design:

- discovery is separate from session construction
- encryption happens client-side
- decryption happens client-side
- the server operates on encrypted payloads

## The same flow via the high-level convenience client

In [ ]:
round_trip = client.infer(model_id=model_id, values=sample_input)
print("High-level client output:")
print(round_trip)